<a href="https://colab.research.google.com/github/jinofedolin/Interactive-humanoid-walking/blob/main/Interactive_humanoid_walking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# GOOGLE COLAB - STABLE INTERACTIVE HUMANOID WALKING SIMULATOR
# ================================================================
#
# 10 DOF:
#
# LEFT LEG  : hip, knee, ankle
# RIGHT LEG : hip, knee, ankle
# LEFT ARM  : shoulder, elbow
# RIGHT ARM : shoulder, elbow
#
# Hands are calculated by forward kinematics.
#
# Main walking phase:
#
#       phi = 2*pi*f*t
#
# Leg:
#       hip = A sin(phi)
#       knee = knee trajectory
#       ankle = A sin(phi)
#
# Arm:
#       opposite phase to corresponding leg
#
# ================================================================

from IPython.display import HTML, display

html = r"""
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<style>

body{
    margin:0;
    padding:15px;
    background:#eef2f5;
    font-family:Arial,sans-serif;
}

.container{
    max-width:1250px;
    margin:auto;
    background:white;
    padding:18px;
    border-radius:12px;
    box-shadow:0 3px 12px rgba(0,0,0,.15);
}

h1{
    text-align:center;
    margin:5px;
}

.subtitle{
    text-align:center;
    margin-bottom:15px;
}

.layout{
    display:flex;
    gap:20px;
    flex-wrap:wrap;
}

.visual{
    flex:2;
    min-width:600px;
}

.panel{
    flex:1;
    min-width:300px;
}

canvas{
    width:100%;
    max-width:760px;
    border:1px solid #aaa;
    border-radius:8px;
    background:white;
}

button{
    padding:10px 18px;
    margin:5px;
    border:0;
    border-radius:6px;
    cursor:pointer;
    font-size:15px;
}

.play{
    background:#198754;
    color:white;
}

.pause{
    background:#f0ad00;
    color:white;
}

.reset{
    background:#dc3545;
    color:white;
}

.control{
    background:#f3f5f7;
    padding:10px;
    margin-bottom:9px;
    border-radius:7px;
}

.control label{
    display:block;
    font-weight:bold;
    margin-bottom:4px;
}

input[type=range]{
    width:100%;
}

.data{
    background:#f7f7f7;
    padding:10px;
    border-radius:7px;
    line-height:1.6;
    margin-top:8px;
}

.title{
    font-weight:bold;
    font-size:17px;
    margin-top:12px;
}

.status{
    padding:10px;
    background:#e8f5e9;
    border-radius:7px;
    font-weight:bold;
}

.formula{
    background:#fff7dd;
    padding:10px;
    margin-top:10px;
    border-radius:7px;
    font-family:serif;
    font-size:17px;
}

</style>

</head>

<body>

<div class="container">

<h1>INTERACTIVE HUMANOID WALKING</h1>

<div class="subtitle">
Two Legs + Two Arms + Hands + Forward Walking
</div>

<div class="layout">

<div class="visual">

<canvas id="canvas"
        width="760"
        height="700">
</canvas>

<br>

<button class="play"
        onclick="play()">
▶ PLAY
</button>

<button class="pause"
        onclick="pause()">
⏸ PAUSE
</button>

<button class="reset"
        onclick="resetSim()">
↻ RESET
</button>

</div>


<div class="panel">

<div class="status">
Status:
<span id="status">PAUSED</span>
</div>


<div class="title">
WALKING CONTROLS
</div>


<div class="control">

<label>
Animation Speed:
<span id="speedText">1.0</span>
</label>

<input id="speed"
       type="range"
       min="0.2"
       max="3"
       step="0.1"
       value="1">

</div>


<div class="control">

<label>
Step Length:
<span id="stepText">0.40 m</span>
</label>

<input id="step"
       type="range"
       min="0.10"
       max="0.70"
       step="0.01"
       value="0.40">

</div>


<div class="control">

<label>
Walking Frequency:
<span id="freqText">0.8 Hz</span>
</label>

<input id="freq"
       type="range"
       min="0.3"
       max="1.5"
       step="0.05"
       value="0.8">

</div>


<div class="title">
ARM CONTROLS
</div>


<div class="control">

<label>
Arm Amplitude:
<span id="armText">25°</span>
</label>

<input id="armAmp"
       type="range"
       min="0"
       max="60"
       step="1"
       value="25">

</div>


<div class="control">

<label>
Arm Phase:
<span id="phaseText">0°</span>
</label>

<input id="armPhase"
       type="range"
       min="-90"
       max="90"
       step="1"
       value="0">

</div>


<div class="control">

<label>
Elbow Bend:
<span id="elbowText">15°</span>
</label>

<input id="elbow"
       type="range"
       min="0"
       max="45"
       step="1"
       value="15">

</div>


<div class="control">

<label>

<input id="sync"
       type="checkbox"
       checked>

Synchronize arms with legs

</label>

</div>


<div class="control">

<label>

<input id="elbowAuto"
       type="checkbox"
       checked>

Automatic elbow movement

</label>

</div>


<div class="title">
SIMULATION DATA
</div>


<div class="data">

Time:
<span id="time">0.00</span> s

<br>

Distance:
<span id="distance">0.00</span> m

<br>

Walking velocity:
<span id="velocity">0.00</span> m/s

<br>

Walking phase:
<span id="walkPhase">0</span>°

</div>


<div class="title">
JOINT ANGLES
</div>


<div class="data">

<b>LEFT LEG</b><br>

Hip:
<span id="LH">0</span>°

<br>

Knee:
<span id="LK">0</span>°

<br>

Ankle:
<span id="LA">0</span>°

<br><br>

<b>RIGHT LEG</b><br>

Hip:
<span id="RH">0</span>°

<br>

Knee:
<span id="RK">0</span>°

<br>

Ankle:
<span id="RA">0</span>°

<br><br>

<b>LEFT ARM</b><br>

Shoulder:
<span id="LS">0</span>°

<br>

Elbow:
<span id="LE">0</span>°

<br><br>

<b>RIGHT ARM</b><br>

Shoulder:
<span id="RS">0</span>°

<br>

Elbow:
<span id="RE">0</span>°

</div>


<div class="formula">

Walking phase:

<br>

φ = 2πft

<br><br>

Leg:

<br>

θ<sub>leg</sub> = A sin(φ)

<br><br>

Arm:

<br>

θ<sub>arm</sub> = −A<sub>a</sub>sin(φ+φ<sub>a</sub>)

</div>

</div>

</div>

</div>


<script>


// ============================================================
// CANVAS
// ============================================================

const canvas =
document.getElementById("canvas");

const ctx =
canvas.getContext("2d");


// ============================================================
// ROBOT DIMENSIONS
// ============================================================

const THIGH = 0.45;

const SHANK = 0.45;

const FOOT = 0.20;

const UPPER_ARM = 0.32;

const FORE_ARM = 0.28;

const HAND = 0.12;

const PELVIS_H = 1.25;

const PELVIS_W = 0.30;

const TORSO_H = 0.60;

const TORSO_W = 0.32;

const SHOULDER_W = 0.45;


// ============================================================
// SIMULATION VARIABLES
// ============================================================

let running = false;

let animationID = null;

let previousTime = null;

let simTime = 0;

let distance = 0;


// ============================================================
// ANGLES
// ============================================================

let LH = 0;

let LK = 0;

let LA = 0;

let RH = 0;

let RK = 0;

let RA = 0;

let LS = 0;

let LE = 0;

let RS = 0;

let RE = 0;


// ============================================================
// HELPER FUNCTIONS
// ============================================================

function deg(rad){

    return rad * 180 / Math.PI;

}


function rad(deg){

    return deg * Math.PI / 180;

}


function get(id){

    return document.getElementById(id);

}


function clamp(x,min,max){

    return Math.max(
        min,
        Math.min(max,x)
    );

}


// ============================================================
// SLIDER VALUES
// ============================================================

function speed(){

    return Number(
        get("speed").value
    );

}


function stepLength(){

    return Number(
        get("step").value
    );

}


function frequency(){

    return Number(
        get("freq").value
    );

}


function armAmplitude(){

    return rad(
        Number(
            get("armAmp").value
        )
    );

}


function armPhase(){

    return rad(
        Number(
            get("armPhase").value
        )
    );

}


function elbowAmount(){

    return rad(
        Number(
            get("elbow").value
        )
    );

}


// ============================================================
// UPDATE SLIDER TEXT
// ============================================================

get("speed").oninput = function(){

    get("speedText").innerText =
        this.value;

};


get("step").oninput = function(){

    get("stepText").innerText =
        Number(this.value).toFixed(2)
        + " m";

};


get("freq").oninput = function(){

    get("freqText").innerText =
        this.value
        + " Hz";

};


get("armAmp").oninput = function(){

    get("armText").innerText =
        this.value
        + "°";

};


get("armPhase").oninput = function(){

    get("phaseText").innerText =
        this.value
        + "°";

};


get("elbow").oninput = function(){

    get("elbowText").innerText =
        this.value
        + "°";

};


// ============================================================
// CALCULATE WALKING JOINT ANGLES
// ============================================================

function calculateAngles(){

    const f =
        frequency();


    const phi =
        2*Math.PI*f*simTime;


    // --------------------------------------------------------
    // HIP
    // --------------------------------------------------------

    const hipAmp =
        rad(18);


    LH =
        hipAmp *
        Math.sin(phi);


    RH =
        hipAmp *
        Math.sin(phi + Math.PI);


    // --------------------------------------------------------
    // KNEE
    //
    // Knee bends mainly during swing phase.
    // --------------------------------------------------------

    const leftSwing =
        Math.max(
            0,
            Math.sin(phi)
        );


    const rightSwing =
        Math.max(
            0,
            Math.sin(phi + Math.PI)
        );


    const kneeMin =
        rad(5);


    const kneeMax =
        rad(48);


    LK =
        kneeMin
        +
        (kneeMax-kneeMin)
        *
        leftSwing
        *
        leftSwing;


    RK =
        kneeMin
        +
        (kneeMax-kneeMin)
        *
        rightSwing
        *
        rightSwing;


    // --------------------------------------------------------
    // ANKLE
    // --------------------------------------------------------

    const ankleAmp =
        rad(10);


    LA =
        -ankleAmp *
        Math.sin(phi);


    RA =
        -ankleAmp *
        Math.sin(phi + Math.PI);


    // --------------------------------------------------------
    // ARMS
    // --------------------------------------------------------

    let armPhi;


    if(
        get("sync").checked
    ){

        armPhi =
            phi +
            armPhase();

    }

    else{

        armPhi =
            2*Math.PI*
            0.8*
            simTime
            +
            armPhase();

    }


    const A =
        armAmplitude();


    // --------------------------------------------------------
    // SHOULDER
    //
    // Opposite arm/leg motion
    // --------------------------------------------------------

    LS =
        -A *
        Math.sin(armPhi);


    RS =
        -A *
        Math.sin(
            armPhi +
            Math.PI
        );


    // --------------------------------------------------------
    // ELBOW
    // --------------------------------------------------------

    if(
        get("elbowAuto").checked
    ){

        const E =
            elbowAmount();


        LE =
            E *
            (
                0.5 +
                0.5 *
                Math.max(
                    0,
                    Math.sin(
                        armPhi
                    )
                )
            );


        RE =
            E *
            (
                0.5 +
                0.5 *
                Math.max(
                    0,
                    Math.sin(
                        armPhi+
                        Math.PI
                    )
                )
            );

    }

    else{

        LE =
            elbowAmount();

        RE =
            elbowAmount();

    }

}


// ============================================================
// LEG FORWARD KINEMATICS
// ============================================================

function legFK(
    hipX,
    hipZ,
    hip,
    knee,
    ankle
){

    const A1 =
        hip;


    const A2 =
        hip +
        knee;


    const A3 =
        hip +
        knee +
        ankle;


    const x0 =
        hipX;


    const z0 =
        hipZ;


    const x1 =
        x0 +
        THIGH *
        Math.sin(A1);


    const z1 =
        z0 -
        THIGH *
        Math.cos(A1);


    const x2 =
        x1 +
        SHANK *
        Math.sin(A2);


    const z2 =
        z1 -
        SHANK *
        Math.cos(A2);


    const x3 =
        x2 +
        FOOT *
        Math.cos(A3);


    const z3 =
        z2 +
        FOOT *
        Math.sin(A3);


    return {

        x:[x0,x1,x2,x3],

        z:[z0,z1,z2,z3]

    };

}


// ============================================================
// ARM FORWARD KINEMATICS
// ============================================================

function armFK(
    shoulderX,
    shoulderZ,
    shoulder,
    elbow,
    side
){

    const A1 =
        shoulder;


    const A2 =
        shoulder +
        elbow;


    const x0 =
        shoulderX;


    const z0 =
        shoulderZ;


    const x1 =
        x0 +
        side *
        UPPER_ARM *
        Math.sin(A1);


    const z1 =
        z0 -
        UPPER_ARM *
        Math.cos(A1);


    const x2 =
        x1 +
        side *
        FORE_ARM *
        Math.sin(A2);


    const z2 =
        z1 -
        FORE_ARM *
        Math.cos(A2);


    const x3 =
        x2 +
        side *
        HAND *
        Math.sin(A2);


    const z3 =
        z2 -
        HAND *
        Math.cos(A2);


    return {

        x:[x0,x1,x2,x3],

        z:[z0,z1,z2,z3]

    };

}


// ============================================================
// WORLD → SCREEN
// ============================================================

function screenX(x){

    return 380 +
        (x-distance) *
        270;

}


function screenY(z){

    return 650 -
        z *
        270;

}


// ============================================================
// DRAW LINE
// ============================================================

function line(
    x1,
    y1,
    x2,
    y2,
    width,
    style
){

    ctx.beginPath();

    ctx.moveTo(x1,y1);

    ctx.lineTo(x2,y2);

    ctx.lineWidth =
        width;

    ctx.strokeStyle =
        style;

    ctx.lineCap =
        "round";

    ctx.stroke();

}


// ============================================================
// DRAW CIRCLE
// ============================================================

function circle(
    x,
    y,
    r
){

    ctx.beginPath();

    ctx.arc(
        x,
        y,
        r,
        0,
        2*Math.PI
    );

    ctx.fillStyle =
        "white";

    ctx.fill();

    ctx.strokeStyle =
        "#333";

    ctx.lineWidth =
        3;

    ctx.stroke();

}


// ============================================================
// DRAW FOOT
// ============================================================

function drawFoot(
    x,
    z
){

    line(
        screenX(x)-5,
        screenY(z),
        screenX(x)+40,
        screenY(z),
        9,
        "#333"
    );

}


// ============================================================
// DRAW ARM
// ============================================================

function drawArm(
    arm
){

    for(
        let i=0;
        i<3;
        i++
    ){

        line(
            screenX(arm.x[i]),
            screenY(arm.z[i]),
            screenX(arm.x[i+1]),
            screenY(arm.z[i+1]),
            14,
            "#8e44ad"
        );

    }


    for(
        let i=0;
        i<4;
        i++
    ){

        circle(
            screenX(arm.x[i]),
            screenY(arm.z[i]),
            8
        );

    }

}


// ============================================================
// DRAW LEG
// ============================================================

function drawLeg(
    leg
){

    for(
        let i=0;
        i<3;
        i++
    ){

        line(
            screenX(leg.x[i]),
            screenY(leg.z[i]),
            screenX(leg.x[i+1]),
            screenY(leg.z[i+1]),
            17,
            "#1769aa"
        );

    }


    for(
        let i=0;
        i<4;
        i++
    ){

        circle(
            screenX(leg.x[i]),
            screenY(leg.z[i]),
            9
        );

    }

}


// ============================================================
// DRAW HUMANOID
// ============================================================

function drawRobot(){

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    // --------------------------------------------------------
    // BACKGROUND
    // --------------------------------------------------------

    ctx.fillStyle =
        "white";

    ctx.fillRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    // --------------------------------------------------------
    // GRID
    // --------------------------------------------------------

    ctx.strokeStyle =
        "#e5e5e5";

    ctx.lineWidth = 1;


    for(
        let x=-2;
        x<=2;
        x+=0.2
    ){

        ctx.beginPath();

        ctx.moveTo(
            screenX(x),
            40
        );

        ctx.lineTo(
            screenX(x),
            650
        );

        ctx.stroke();

    }


    for(
        let z=0;
        z<=2.5;
        z+=0.2
    ){

        ctx.beginPath();

        ctx.moveTo(
            0,
            screenY(z)
        );

        ctx.lineTo(
            760,
            screenY(z)
        );

        ctx.stroke();

    }


    // --------------------------------------------------------
    // GROUND
    // --------------------------------------------------------

    line(
        0,
        screenY(0),
        760,
        screenY(0),
        5,
        "#222"
    );


    // --------------------------------------------------------
    // BODY
    // --------------------------------------------------------

    const pelvisZ =
        PELVIS_H;


    const leftHipX =
        distance -
        PELVIS_W/2;


    const rightHipX =
        distance +
        PELVIS_W/2;


    const shoulderZ =
        PELVIS_H +
        TORSO_H +
        0.05;


    const leftShoulderX =
        distance -
        SHOULDER_W/2;


    const rightShoulderX =
        distance +
        SHOULDER_W/2;


    // --------------------------------------------------------
    // LEGS
    // --------------------------------------------------------

    const leftLeg =
        legFK(
            leftHipX,
            pelvisZ,
            LH,
            LK,
            LA
        );


    const rightLeg =
        legFK(
            rightHipX,
            pelvisZ,
            RH,
            RK,
            RA
        );


    // --------------------------------------------------------
    // ARMS
    // --------------------------------------------------------

    const leftArm =
        armFK(
            leftShoulderX,
            shoulderZ,
            LS,
            LE,
            -1
        );


    const rightArm =
        armFK(
            rightShoulderX,
            shoulderZ,
            RS,
            RE,
            1
        );


    // --------------------------------------------------------
    // DRAW LEGS
    // --------------------------------------------------------

    drawLeg(
        leftLeg
    );

    drawLeg(
        rightLeg
    );


    // --------------------------------------------------------
    // DRAW ARMS
    // --------------------------------------------------------

    drawArm(
        leftArm
    );

    drawArm(
        rightArm
    );


    // --------------------------------------------------------
    // PELVIS
    // --------------------------------------------------------

    ctx.fillStyle =
        "#555";

    ctx.fillRect(
        screenX(
            distance -
            PELVIS_W/2
        ),
        screenY(
            PELVIS_H+
            0.07
        ),
        PELVIS_W*270,
        0.14*270
    );


    // --------------------------------------------------------
    // TORSO
    // --------------------------------------------------------

    ctx.fillStyle =
        "#707070";

    ctx.fillRect(
        screenX(
            distance -
            TORSO_W/2
        ),
        screenY(
            PELVIS_H+
            TORSO_H
        ),
        TORSO_W*270,
        TORSO_H*270
    );


    // --------------------------------------------------------
    // HEAD
    // --------------------------------------------------------

    ctx.beginPath();

    ctx.arc(
        screenX(distance),
        screenY(
            PELVIS_H+
            TORSO_H+
            0.16
        ),
        28,
        0,
        2*Math.PI
    );

    ctx.fillStyle =
        "#888";

    ctx.fill();

    ctx.strokeStyle =
        "#444";

    ctx.lineWidth =
        3;

    ctx.stroke();


    // --------------------------------------------------------
    // CoM
    // --------------------------------------------------------

    const comZ =
        PELVIS_H+
        TORSO_H*0.40;


    ctx.beginPath();

    ctx.arc(
        screenX(distance),
        screenY(comZ),
        8,
        0,
        2*Math.PI
    );

    ctx.fillStyle =
        "#d9534f";

    ctx.fill();


    ctx.fillStyle =
        "#333";

    ctx.font =
        "15px Arial";

    ctx.fillText(
        "CoM",
        screenX(distance)+12,
        screenY(comZ)
    );


    // --------------------------------------------------------
    // FEET
    // --------------------------------------------------------

    drawFoot(
        leftLeg.x[3],
        leftLeg.z[3]
    );


    drawFoot(
        rightLeg.x[3],
        rightLeg.z[3]
    );


    // --------------------------------------------------------
    // HAND LABELS
    // --------------------------------------------------------

    ctx.fillStyle =
        "#333";

    ctx.font =
        "14px Arial";


    ctx.fillText(
        "L Hand",
        screenX(leftArm.x[3])-45,
        screenY(leftArm.z[3])-10
    );


    ctx.fillText(
        "R Hand",
        screenX(rightArm.x[3])+10,
        screenY(rightArm.z[3])-10
    );


    // --------------------------------------------------------
    // WALKING ARROW
    // --------------------------------------------------------

    ctx.font =
        "bold 16px Arial";

    ctx.fillText(
        "WALKING →",
        20,
        680
    );

}


// ============================================================
// UPDATE DATA
// ============================================================

function updateData(){

    get("time").innerText =
        simTime.toFixed(2);


    get("distance").innerText =
        distance.toFixed(2);


    get("velocity").innerText =
        (
            stepLength()*
            frequency()
        ).toFixed(2);


    let phi =
        (
            2*Math.PI*
            frequency()*
            simTime
        )
        %
        (2*Math.PI);


    get("walkPhase").innerText =
        deg(phi).toFixed(0);


    get("LH").innerText =
        deg(LH).toFixed(1);

    get("LK").innerText =
        deg(LK).toFixed(1);

    get("LA").innerText =
        deg(LA).toFixed(1);


    get("RH").innerText =
        deg(RH).toFixed(1);

    get("RK").innerText =
        deg(RK).toFixed(1);

    get("RA").innerText =
        deg(RA).toFixed(1);


    get("LS").innerText =
        deg(LS).toFixed(1);

    get("LE").innerText =
        deg(LE).toFixed(1);


    get("RS").innerText =
        deg(RS).toFixed(1);

    get("RE").innerText =
        deg(RE).toFixed(1);

}


// ============================================================
// ANIMATION
// ============================================================

function animate(timestamp){

    if(!running){

        return;

    }


    if(
        previousTime === null
    ){

        previousTime =
            timestamp;

    }


    let dt =
        (
            timestamp -
            previousTime
        ) / 1000;


    previousTime =
        timestamp;


    // Avoid large time jump

    dt =
        Math.min(
            dt,
            0.04
        );


    // Simulation speed

    dt *=
        speed();


    // --------------------------------------------------------
    // TIME
    // --------------------------------------------------------

    simTime += dt;


    // --------------------------------------------------------
    // FORWARD MOTION
    //
    // v = step length × frequency
    // --------------------------------------------------------

    distance +=
        stepLength()*
        frequency()*
        dt;


    // --------------------------------------------------------
    // JOINT TRAJECTORIES
    // --------------------------------------------------------

    calculateAngles();


    // --------------------------------------------------------
    // DRAW
    // --------------------------------------------------------

    drawRobot();


    updateData();


    animationID =
        requestAnimationFrame(
            animate
        );

}


// ============================================================
// PLAY
// ============================================================

function play(){

    if(running){

        return;

    }


    running =
        true;


    previousTime =
        null;


    get("status").innerText =
        "RUNNING";


    animationID =
        requestAnimationFrame(
            animate
        );

}


// ============================================================
// PAUSE
// ============================================================

function pause(){

    running =
        false;


    if(animationID){

        cancelAnimationFrame(
            animationID
        );

    }


    animationID =
        null;


    previousTime =
        null;


    get("status").innerText =
        "PAUSED";

}


// ============================================================
// RESET
// ============================================================

function resetSim(){

    running =
        false;


    if(animationID){

        cancelAnimationFrame(
            animationID
        );

    }


    animationID =
        null;


    previousTime =
        null;


    simTime =
        0;


    distance =
        0;


    LH = 0;

    LK = 0;

    LA = 0;

    RH = 0;

    RK = 0;

    RA = 0;

    LS = 0;

    LE = 0;

    RS = 0;

    RE = 0;


    get("status").innerText =
        "RESET";


    calculateAngles();

    drawRobot();

    updateData();

}


// ============================================================
// INITIALIZE
// ============================================================

calculateAngles();

drawRobot();

updateData();


</script>

</body>

</html>
"""

display(HTML(html))